In [2]:
import params
import illustris_python as il
import numpy as np
import Catalogue
import pickle
import matplotlib.mlab as mlab

In [3]:
basePath = '/home/tnguser/sims.TNG/TNG100-1/output/'

In [4]:
params.N_list

[99]

In [5]:
for N in params.N_list:
    print(f'Now starts the processing of the snapshot {N}')
    Header = il.groupcat.loadHeader(basePath, N)
    Halos_cat = il.groupcat.loadHalos(basePath,N,fields=params.Halos_Fields )
    Subhalos_cat = il.groupcat.loadSubhalos(basePath, N, fields= params.Subhalos_Fields)
    print(f'Catalogues of snapshot {N} loaded.')
    
    selected_halos = Catalogue.select_halos(Halos_cat, params.Halos_limits)
    selected_subhalos = Catalogue.select_subhalos(Subhalos_cat, params.Subhalos_limits, selected_halos)
    print(f"N halos in {N} : {np.size(np.where(selected_halos))}")
    print(f"N subhalos in {N} : {np.size(np.where(selected_subhalos))}")
    
    Halos_cat, Subhalos_cat = Catalogue.TNG2PROPnum(Halos_cat, Subhalos_cat, N)
    aux_Halos_cat = Catalogue.dictionary_select_items(Halos_cat, selected_halos)
    aux_Halos_cat = Catalogue.add_center_mass(aux_Halos_cat, basePath, N, Header, Halos_cat, selected_halos, 'gas')
    aux_Halos_cat['GroupOffset'] = np.linalg.norm(Catalogue.Distance_3D_vectors(aux_Halos_cat['GroupSZPos'], aux_Halos_cat['GroupPos'], Header['BoxSize']), axis=1)/aux_Halos_cat['Group_R_Crit200']
    aux_Halos_cat = Catalogue.add_center_mass_DM(aux_Halos_cat, basePath, N, Header, Halos_cat, selected_halos)
    aux_Subhalos_cat = Catalogue.dictionary_select_items(Subhalos_cat.copy(), selected_subhalos)
   
    print("Calculating gas at differents temperatures ... ")
    aux_Subhalos_cat['SubhaloMassTemp']=Catalogue.add_SubhaloMassTemp(Subhalos_cat, selected_subhalos, N, basePath)
    print("Finish")
    
    aux_Subhalos_cat['Subhalo_CoA'], aux_Subhalos_cat['Subhalo_IT'], aux_Subhalos_cat['Subhalo_NumStarPart']=Catalogue.add_InertiaTensor(basePath,Subhalos_cat,selected_subhalos, N, Header, 'SubhaloHalfmassRadType',2 )
    aux_Subhalos_cat['Subhalo_QS'], aux_Subhalos_cat['Subhalo_RIT'], aux_Subhalos_cat['Subhalo_NumStarPart_RT']=Catalogue.add_ReducedInertiaTensor(basePath,Subhalos_cat,selected_subhalos, N, Header, 'SubhaloHalfmassRadType',2 )
    
    if 'Halos' in locals():
        Halos=Catalogue.join_dictionaries(Halos, aux_Halos_cat)
        Subhalos=Catalogue.join_dictionaries(Subhalos, aux_Subhalos_cat)
        print("Se realizó la unión de los catálogos")
    else:
        Halos=aux_Halos_cat
        Subhalos=aux_Subhalos_cat
        print("Se creo los dictionarios Halos y Subhalos")
        
print(Halos['GroupNsubs'])
print("Calculating the distance ... ")
Subhalos, Halos=Catalogue.add_SubhalosDist2(Subhalos, Halos, Header)
print('Finish')

Subhalos['SubhaloFlag'][np.in1d(Subhalos['SubhaloNumber'], Halos['GroupFirstSub'])]=False
Subhalos['SubhaloLogStellarMass']=np.log10(Subhalos['SubhaloMassType'][:,4]*10**(10)/Header['HubbleParam'])
Subhalos['SubhalosSFR']=np.divide(Subhalos['SubhaloSFR'],Subhalos['SubhaloMassType'][:,4]*10**10/Header['HubbleParam'])

print(f"Se actualizaron {np.size(np.where(Subhalos['SubhalosSFR']==0)[0])} halos con sSFR 0 por el valor de {np.min(Subhalos['SubhalosSFR'])}")
#Subhalos['SubhalosSFR'][np.where(Subhalos['SubhalosSFR']==0)[0]]=np.min(Subhalos['SubhalosSFR'][np.nonzero(Subhalos['SubhalosSFR'])])
Subhalos['SubhalosSFR'][np.where(Subhalos['SubhalosSFR']==0)[0]]=8E-15
Subhalos['SubhaloLogsSFR']=np.log10(Subhalos['SubhalosSFR'])
Subhalos['Redshift']=np.array([Catalogue.get_redshift(GrNr) for GrNr in Subhalos['SubhaloGrNr']])
Subhalos['SubhaloQuenchLim']=Catalogue.quench_value(Subhalos['SubhaloLogStellarMass'], Subhalos['Redshift'])

print("Hey.... This program works well !! (apparently)")


Now starts the processing of the snapshot 99
Catalogues of snapshot 99 loaded.
N halos in 99 : 172
N subhalos in 99 : 4729
Computing the center of mass of the gas in the snapshot 99
Computing the center of mass of the dark matter in the snapshot 99
Calculating gas at differents temperatures ... 
[[3.89205298e+03 4.94559479e+00 3.08079958e+00 4.82653081e-01
  3.90056201e+03]
 [4.37232422e+02 3.87224150e+00 2.62145567e+00 8.16314697e-01
  4.44542450e+02]
 [2.52119160e+00 5.03453493e-01 9.19904709e+00 3.87940526e-01
  1.26116323e+01]
 ...
 [3.95112839e+01 1.29040420e+00 3.14615297e+00 8.38686824e-01
  4.47865295e+01]
 [8.85748246e-04 2.60550785e-03 2.64486223e-02 1.99976814e-04
  3.01398560e-02]
 [0.00000000e+00 1.95077471e-02 1.96631178e-01 1.10983565e-01
  3.27122509e-01]]
Finish
Computing the inertia tensor 
Computing the inertia tensor 
Se creo los dictionarios Halos y Subhalos
[17185 14157 10240 11036  8113  8776  6579  7194  5383  7837  6183  5329
  6376  4291  3184  3164  3366  265

In [6]:
HF = open(params.OHalos, "wb")
pickle.dump(Halos, HF)
HF.close()

SHF = open(params.OSubhalos, "wb")
pickle.dump(Subhalos, SHF)
SHF.close()
print("Files saved correctly !!")

Files saved correctly !!
